# Algoritma Genetika: Studi Kasus Word Matching

In [1]:
import random
import numpy as np
import matplotlib.pyplot as plt

Target_word = "GENETIKA"
target = [ord(c) - ord('A') for c in Target_word]

print("Target word:", Target_word)
print("Target word (numerical):", target)
print("panjang target:", len(target))

Target word: GENETIKA
Target word (numerical): [6, 4, 13, 4, 19, 8, 10, 0]
panjang target: 8


# Parameter Algoritma Genetika

In [2]:
jumlah_populasi = 100
jumlah_generasi = 10
prob_crossover = 0.7
prob_mutasi = 0.01
panjang_gen = len(target)
min_allele = 1
max_allele = 26
max_fitness = max_allele * panjang_gen

print("Parameter Algoritma Genetika:")
print("Jumlah populasi:", jumlah_populasi)
print("Jumlah generasi:", jumlah_generasi)
print("Probabilitas crossover:", prob_crossover)
print("Probabilitas mutasi:", prob_mutasi)
print("Panjang gen:", panjang_gen)
print("Rentang allele:", min_allele, "sampai", max_allele)
print("Fitness maksimal:", max_fitness)

Parameter Algoritma Genetika:
Jumlah populasi: 100
Jumlah generasi: 10
Probabilitas crossover: 0.7
Probabilitas mutasi: 0.01
Panjang gen: 8
Rentang allele: 1 sampai 26
Fitness maksimal: 208


# Fungsi Helper

In [4]:
def angka_ke_huruf(individu):
    """
    Mengubah list angka menjadi string kata.
    Contoh: [7,5,14,5,20,9,11,1] -> 'GENETIKA'
    """
    return ''.join([chr(g + ord('A') - 1) for g in individu])

# Uji coba fungsi
print("Uji angka_ke_huruf:")
print(f"  {target} -> '{angka_ke_huruf(target)}'")
print(f"  [1,2,3] -> '{angka_ke_huruf([1,2,3])}'")

Uji angka_ke_huruf:
  [6, 4, 13, 4, 19, 8, 10, 0] -> 'FDMDSHJ@'
  [1,2,3] -> 'ABC'


# Pembangkit Populasi Awal

In [ ]:
def create_individual():
    return [random.randint(min_allele, max_allele) for _ in range(panjang_gen)]

def create_population(jumlah_populasi):
    return [create_individual() for _ in range(jumlah_populasi)]

populasi_example = create_population(3)
print("Contoh individu dalam populasi:")

for i, individu in enumerate(populasi_example):
    print(f'Individu {i+1}: {individu} -> {angka_ke_huruf(individu)}', individu)

Contoh individu dalam populasi:
Individu 1: [15, 5, 13, 26, 14, 10, 24, 5] -> OEMZNJXE [15, 5, 13, 26, 14, 10, 24, 5]
Individu 2: [12, 1, 26, 8, 10, 5, 17, 8] -> LAZHJEQH [12, 1, 26, 8, 10, 5, 17, 8]
Individu 3: [12, 2, 18, 11, 15, 4, 7, 5] -> LBRKODGE [12, 2, 18, 11, 15, 4, 7, 5]


# Fungsi Fitness

In [21]:
def calculate_fitness(individu,target):
    total_selisih = sum(abs(individu[i] - target[i]) for i in range(len(target)))
    fitness = max_fitness - total_selisih
    return fitness

def calculate_fitness_all(population, target):
    return [calculate_fitness(individu, target) for individu in population]

individu_example = create_individual()
fitness_example = calculate_fitness(individu_example, target)
print("Uji Fungsi Fitness (contoh dari slide):")
print(f"  Individu : {individu_example} = '{angka_ke_huruf(individu_example)}'")
print(f"  Target   : {target} = '{angka_ke_huruf(target)}'")
print(f"  Fitness  : {max_fitness} - {sum(abs(individu_example[i] - target[i]) for i in range(len(target)))} = {fitness_example}  (harusnya 162)")
print()
print(f"  Fitness TARGET itu sendiri: {calculate_fitness(target, target)}  (harusnya {max_fitness})")

Uji Fungsi Fitness (contoh dari slide):
  Individu : [25, 16, 22, 19, 3, 1, 5, 4] = 'YPVSCAED'
  Target   : [6, 4, 13, 4, 19, 8, 10, 0] = 'FDMDSHJ@'
  Fitness  : 208 - 87 = 121  (harusnya 162)

  Fitness TARGET itu sendiri: 208  (harusnya 208)


# Fungsi Seleksi (Roullete Wheel)

In [26]:
def select_roullette(population, fitness_values):
    total_fitness = sum(fitness_values)
    if total_fitness == 0:
        return random.choice(population)
    pick = random.uniform(0, total_fitness)
    current = 0
    for individu, fitness in zip(population, fitness_values):
        current += fitness
        if current > pick:
            return individu
    
    return population[-1]


print("Fungsi seleksi_roulette ")
print()
print("Ilustrasi: individu dengan fitness lebih tinggi")
print("lebih sering muncul saat dipilih berkali-kali:")
pop_kecil   = [[1]*8, [13]*8, [7,5,14,5,20,9,11,1]]  # buruk, sedang, sempurna
fit_kecil   = [calculate_fitness(p, target) for p in pop_kecil]
for p, f in zip(pop_kecil, fit_kecil):
    print(f"  {angka_ke_huruf(p)} -> fitness {f}")

Fungsi seleksi_roulette 

Ilustrasi: individu dengan fitness lebih tinggi
lebih sering muncul saat dipilih berkali-kali:
  AAAAAAAA -> fitness 150
  MMMMMMMM -> fitness 156
  GENETIKA -> fitness 200


# Crossover

menggabungkan dua parent untuk membuat anak baru.

In [ ]:
def crossover(parent1, parent2, prob_co):
    anak1 = parent1.copy()
    anak2 = parent2.copy()
    
    if random.random() < prob_co:
        titik1 = random.randint(1, panjang_gen - 2)
        titik2 = random.randint(titik1 + 1, panjang_gen - 1)
        
        anak1[titik1:titik2] = parent2[titik1:titik2]
        anak2[titik1:titik2] = parent1[titik1:titik2]
        
    return anak1, anak2

individu1 = [14, 6, 2, 4, 23, 15, 17, 8]
individu2 = [12, 22, 7, 13, 11, 6, 4, 16]

anak1, anak2 = crossover(individu1, individu2, prob_co=1.0)  # paksa crossover
print("Uji Fungsi Crossover:")
print(f"  Parent 1: {individu1} -> '{angka_ke_huruf(individu1)}'")
print(f"  Parent 2: {individu2} -> '{angka_ke_huruf(individu2)}'")
print(f"  Anak 1   : {anak1} -> '{angka_ke_huruf(anak1)}'")
print(f"  Anak 2   : {anak2} -> '{angka_ke_huruf(anak2)}'")


Uji Fungsi Crossover:
  Parent 1: [14, 6, 2, 4, 23, 15, 17, 8] -> 'NFBDWOQH'
  Parent 2: [12, 22, 7, 13, 11, 6, 4, 16] -> 'LVGMKFDP'
  Anak 1   : [14, 6, 2, 13, 11, 15, 17, 8] -> 'NFBMKOQH'
  Anak 2   : [12, 22, 7, 4, 23, 6, 4, 16] -> 'LVGDWFDP'


# Mutation

ubah sedikit nilai gen secara acak 

In [ ]:
def mutation(individu, prob_mutasi):
   
   hasil = individu.copy()
   if random.random() < prob_mutasi:
    posisi = random.randint(0, panjang_gen - 1)
    delta = random.choice([-5,-4,-3,-2,-1,1,2,3,4,5])
    nilai_baru = hasil[posisi] + delta
    nilai_baru = max(min_allele, min(max_allele, nilai_baru))
    hasil[posisi] = nilai_baru
   return hasil


individu_sebelum =  [8, 5, 14, 11, 19, 6, 11, 1]
individu_setelah = mutation(individu_sebelum, prob_mutasi=1.0)  
print("Uji Fungsi Mutasi:")
print(f"  Sebelum: {individu_sebelum} -> '{angka_ke_huruf(individu_sebelum)}'")
print(f"  Setelah : {individu_setelah} -> '{angka_ke_huruf(individu_setelah)}'")


Uji Fungsi Mutasi:
  Sebelum: [8, 5, 14, 11, 19, 6, 11, 1] -> 'HENKSFKA'
  Setelah : [8, 5, 14, 16, 19, 6, 11, 1] -> 'HENPSFKA'


# Elitism

gabungan induk + anak, lalu diambil yang n terbaik.

In [45]:
def elitism(new_population, old_population, target, n ):
    gabungan = new_population + old_population
    
    fitness_gabungan = calculate_fitness_all(gabungan, target)
    sorted_indices = np.argsort(fitness_gabungan)[::-1]
    return [gabungan[i] for i in sorted_indices[:n]]

print("Uji Fungsi Elitism:")
pop_lama = [[1]*8, [13]*8, [7,5,14,5,20,9,11,1]]  # buruk, sedang, sempurna
pop_baru = [[2]*8, [12]*8, [7,5,14,5,20,9,11,1]]  # sedikit lebih baik, sedikit lebih buruk, sempurna
elit = elitism(pop_baru, pop_lama, target, n=3) 
print("Populasi Lama:")
for p in pop_lama:
    print(f"  {angka_ke_huruf(p)} -> fitness {calculate_fitness(p, target)}")
print("Populasi Baru:")
for p in pop_baru:
    print(f"  {angka_ke_huruf(p)} -> fitness {calculate_fitness(p, target)}")
print("Populasi Elit:")
for p in elit:
    print(f"  {angka_ke_huruf(p)} -> fitness {calculate_fitness(p, target)}")

Uji Fungsi Elitism:
Populasi Lama:
  AAAAAAAA -> fitness 150
  MMMMMMMM -> fitness 156
  GENETIKA -> fitness 200
Populasi Baru:
  BBBBBBBB -> fitness 156
  LLLLLLLL -> fitness 160
  GENETIKA -> fitness 200
Populasi Elit:
  GENETIKA -> fitness 200
  GENETIKA -> fitness 200
  LLLLLLLL -> fitness 160
